In [ ]:
# mike babb
# created: 2026 08 23
# find five groups of five letters

In [ ]:
# standard
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx

# FUNCTIONS

In [ ]:
# define a function to load a pickle
def load_pickle(file_name):
    if os.path.exists(file_name):
        with open(file_name, 'rb') as handle:
            de_pickle = pickle.load(handle)
    else:
        de_pickle = None
        print("file does not exist")
    return de_pickle

In [ ]:
def get_vowels(letters, r_word, min_vowel_count, existing_words):    

    # combine the existing letters and the remaining word
    letter_group = set(letters + r_word)
    rg_vowels = vowel_set.difference(letter_group)
    rg_vowel_count = len(rg_vowels)    

    if rg_vowel_count >= min_vowel_count:        
        letter_group = ''.join(sorted(letter_group))    
        remainder_group  = lc_set.difference(letter_group)
        remainder_group = ''.join(sorted(remainder_group))
        temp_list = [r_word, letter_group, remainder_group]
        existing_words.extend(temp_list)
        return existing_words
    else:
        return None

# LOAD LETTERS

In [ ]:
letter_dict = load_pickle(file_name = 'letter_dict.pkl')

In [ ]:
word_df = pd.read_csv(filepath_or_buffer=  'words_alpha.txt', header = None, names = ['word'], dtype = str)

In [ ]:
word_df['word'] = word_df['word'].astype(str)

In [ ]:
word_df.head()

In [ ]:
word_df.shape

In [ ]:
word_df['word'].isna().value_counts()

In [ ]:
word_df['lcase'] = word_df['word'].str.lower()

In [ ]:
word_df['n_letters'] = word_df['word'].str.len()

In [ ]:
word_df['letters_sorted'] = word_df['lcase'].map(lambda x: ''.join(sorted(x)))

In [ ]:
word_df['lcase_set'] = word_df['lcase'].map(lambda x: set(x))
word_df['lcase_tuple'] = word_df['lcase_set'].map(lambda x: tuple(x))

In [ ]:
word_df['n_unique_chars'] = word_df['lcase_set'].map(lambda x: len(x))

In [ ]:
word_df = word_df.loc[(word_df['n_unique_chars'] == 5) & (word_df['n_letters'] == 5), :]
word_df = word_df.sort_values(by = 'lcase').reset_index(drop = True)

In [ ]:
# using the word_group, select entries
word_df = word_df.drop_duplicates(subset = ['letters_sorted'])

In [ ]:
word_df.head()

In [ ]:
word_df.shape

In [ ]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

In [ ]:
word_df['word_id'] = range(0, word_df.shape[0])

In [ ]:
word_df.shape

In [ ]:
word_df.head()

In [ ]:
lc_wid_dict = {lcase:word_id for lcase, word_id in zip(word_df['lcase'], word_df['word_id'])}
wid_lc_dict = {word_id:lcase for lcase, word_id in zip(word_df['lcase'], word_df['word_id'])}

In [ ]:
word_id_list = word_df['word_id'].to_numpy(dtype = np.int16)

In [ ]:
working_word_df = word_df[['word_id', 'lcase']].copy()

In [ ]:
# build dictionaries
word_dict = {}
for i_row, my_row in word_df.iterrows():
    word_dict[my_row['lcase']] = my_row['lcase_set']

In [ ]:
lc_set = set(ascii_lowercase)

In [ ]:
vowel_set = set('aeiouy')

# BUILD A CHAR MATRIX

In [ ]:
# build a char_matrix
char_matrix = np.zeros(shape = (word_df.shape[0], 26), dtype = np.int8)
def build_char_matrix(row):
    word = sorted(row['lcase'])
    for iw, w in enumerate(word):
        char_matrix[row['word_id'], letter_dict[w]] += 1


In [ ]:
outcome = word_df.apply(build_char_matrix, axis = 1)

In [ ]:
assert word_df.shape[0] == char_matrix.shape[0]

In [ ]:
char_matrix.shape

# FIND WORDS WITHOUT VOWELS

In [ ]:
vowel_index = [letter_dict[v] for v in vowel_set]
no_vowel_idx = (char_matrix[:, vowel_index] == 0).all(axis = 1)
no_vowel_df = word_df.loc[no_vowel_idx, :].copy()

In [ ]:
no_vowel_cm = char_matrix[no_vowel_idx, :]

In [ ]:
words_without_vowels = no_vowel_df['lcase'].tolist() 

In [ ]:
word_df['no_vowel'] = int(0)

In [ ]:
word_df.loc[word_df['lcase'].isin(words_without_vowels), 'no_vowel'] = 1

In [ ]:
word_df['no_vowel'].sum()

# BUILD LEVEL 2 USING COMBINATIONS

In [ ]:
def byte_encode_words(word):
    ret = 0
    for c in sorted(word):
        alpha_index = ord(c) - ord("a")
        ret |= 1 << alpha_index
    return ret

In [ ]:
word_df['word_byte'] = word_df['word'].map(byte_encode_words)

In [ ]:
word_df.head()

In [ ]:
word_byte_list = word_df['word_byte'].tolist()

In [ ]:
w1 = 'abhor'
w2 = 'cleft'
w3 = 'frown'
w1b = byte_encode_words(w1)
w2b = byte_encode_words(w2)
w3b = byte_encode_words(w3)

In [ ]:
# no letters in common
w1b & w2b

In [ ]:
# letters in common
w1b & w3b

In [ ]:
w1b | w2b

In [ ]:
# this is the same as directly above
testo = byte_encode_words('abhorcleft')
testo

In [ ]:
lc_be = byte_encode_words(ascii_lowercase)

In [ ]:
lc_be

In [ ]:
edge_list = []
for w1_be, w2_be in combinations(word_byte_list, 2):
    if w1_be & w2_be == 0:   
        # they share no letters in common
        existing_letters = w1_be | w2_be                        
        edge_list.append([w1_be, w2_be, existing_letters])


In [ ]:
# export to an edgelist
l2_df = pd.DataFrame(data = edge_list, columns = ['w1', 'w2', 'l2'])

In [ ]:
l2_df.shape

In [ ]:
l2_df = l2_df.drop_duplicates(subset = ['l2']).reset_index(drop = True)

In [ ]:
l2_df.shape

In [ ]:
l2_df.head()

In [ ]:
word_byte_array = np.array(word_byte_list)

In [ ]:
l2_df.head()

In [ ]:
20491 | 264468

In [ ]:
l3_list = np.full(shape = (100000000, 5), fill_value = -1, dtype = int)
start_pos = 0
found_values = set()
for i_row, row in l2_df.iterrows():    
    w1b, w2b, l2 = row
    # indexer
    positional_idx = (word_byte_array & l2) == 0

    # words with different letters
    output_array_w3b = word_byte_array[positional_idx]    

    # accumulated letters
    output_array_l3 = output_array_w3b | l2

    # get the shape to build the output
    n_pairs = output_array_l3.shape[0]
    temp_array = np.zeros(shape = (n_pairs, 5), dtype = int)

    temp_array[:, 0] = w1b
    temp_array[:, 1] = w2b    
    temp_array[:, 2] = output_array_w3b
    temp_array[:, 3] = l2  # (w1b & w2b)
    temp_array[:, 4] = output_array_l3 # (w1b & w2b)
    

    output_array_l3_test = [x not in found_values for x in output_array_l3 ]
    temp_array = temp_array[output_array_l3_test, :]
    found_values.update(temp_array[:, 4])

    n_pairs = temp_array.shape[0]
    end_pos = start_pos + n_pairs
    l3_list[start_pos:end_pos, :] = temp_array
    start_pos = end_pos

    if i_row % 10000 == 0:
        print(i_row)  


In [ ]:
l3_list  = l3_list[:end_pos, :]

In [ ]:
l3_list.shape

In [ ]:
l3_df = pd.DataFrame(data = l3_list, columns = ['w1b', 'w2b', 'w3b', 'l2', 'l3'])

In [ ]:
l3_df.head()

In [ ]:
word_byte_to_word_dict = {wb:lcase for wb, lcase in zip(word_df['word_byte'], word_df['lcase'])}

In [ ]:
word_byte_to_word_dict[l3_df['w1b'].iloc[0]]

In [ ]:
word_byte_to_word_dict[l3_df['w2b'].iloc[0]]

In [ ]:
word_byte_to_word_dict[l3_df['w3b'].iloc[0]]

In [ ]:
byte_encode_words('abdom') | byte_encode_words('ceils') | byte_encode_words('funky')

In [ ]:
l3_df['l3'].unique().shape

In [ ]:
l3_df.head()

In [ ]:
l4_list = np.full(shape = (100000000, 8), fill_value = -1, dtype = int)
start_pos = 0
found_values = set()
for i_row, row in l3_df.iterrows():    
    w1b, w2b, w3b, l2, l3 = row

    # indexer    
    positional_idx = (word_byte_array & l3) == 0
    if positional_idx.size > 0:

        # words with different letters
        output_array_w4b = word_byte_array[positional_idx]    

        # accumulated letters
        output_array_l4 = output_array_w4b | l3

        # get the shape to build the output
        n_pairs = output_array_l4.shape[0]
        temp_array = np.zeros(shape = (n_pairs, 8), dtype = int)

        temp_array[:, 0] = w1b
        temp_array[:, 1] = w2b    
        temp_array[:, 2] = w3b
        temp_array[:, 3] = output_array_w4b
        temp_array[:, 4] = l2
        temp_array[:, 5] = l3  # (w1b | w2b | w3b)
        temp_array[:, 6] = output_array_l4 # (w1b | w2b | w3b | w4b) l4
        

        output_array_l4_test = [x not in found_values for x in output_array_l4 ]
        temp_array = temp_array[output_array_l4_test, :]
        found_values.update(temp_array[:, 6])

        n_pairs = temp_array.shape[0]
        end_pos = start_pos + n_pairs
        l4_list[start_pos:end_pos, :] = temp_array
        start_pos = end_pos

        if i_row % 10000 == 0:
            print(i_row)  


In [ ]:
l4_list = l4_list[:start_pos, :]

In [ ]:
l4_list.shape

In [ ]:
l4_df = pd.DataFrame(data = l4_list, columns = ['w1b', 'w2b', 'w3b', 'w4b', 'l2', 'l3', 'l4'])

In [ ]:
l4_df = l4_df.drop(labels = ['extra'], axis = 1)

In [ ]:
l4_df.head()

In [ ]:
# save stuff....

In [ ]:
l5_list = np.full(shape = (100000000, 9), fill_value = -1, dtype = int)
start_pos = 0
found_values = set()
for i_row, row in l4_df.iterrows():    
    w1b, w2b, w3b, w4b, l2, l3, l4 = row

    # indexer    
    positional_idx = (word_byte_array & l4) == 0
    if positional_idx.sum() > 0:

        # words with different letters
        output_array_w5b = word_byte_array[positional_idx]
        #print(positional_idx)
        #print(output_array_w5b)

        # accumulated letters
        output_array_l5 = output_array_w5b | l4

        # get the shape to build the output
        n_pairs = output_array_l5.shape[0]
        temp_array = np.zeros(shape = (n_pairs, 9), dtype = int)

        temp_array[:, 0] = w1b
        temp_array[:, 1] = w2b    
        temp_array[:, 2] = w3b
        temp_array[:, 3] = w4b
        temp_array[:, 4] = output_array_w5b
        temp_array[:, 5] = l2  # (w1b | w2b)
        temp_array[:, 6] = l3  # (w1b | w2b | w3b)
        temp_array[:, 7] = l4  # (w1b | w2b | w3b | w4b)
        temp_array[:, 8] = output_array_l5 # (w1b | w2b | w3b | w4b | w5b)
        
        
        output_array_l5_test = [x not in found_values for x in output_array_l5 ]
        temp_array = temp_array[output_array_l5_test, :]
        found_values.update(temp_array[:, 8])

        n_pairs = temp_array.shape[0]
        end_pos = start_pos + n_pairs
        l5_list[start_pos:end_pos, :] = temp_array
        start_pos = end_pos

    if i_row % 10000 == 0:
        print(i_row)  


In [ ]:
l5_list = l5_list[:start_pos, :]

In [ ]:
l5_list.shape

In [ ]:
# put it all together

In [ ]:
l5_df = pd.DataFrame(data = l5_list, columns = ['w1b', 'w2b', 'w3b', 'w4b', 'w5b', 'l2', 'l3', 'l4', 'l5'])

In [ ]:
l5_df.head()

In [ ]:
for idx in range(1, 6):
    cn = f"w{str(idx)}b"
    ncn = f"w{str(idx)}"
    l5_df[ncn] = l5_df[cn].map(word_byte_to_word_dict)

In [ ]:
l5_df

In [ ]:
l5_df['l5'].unique().shape

In [ ]:
l2_df.shape

In [ ]:
l3_df.shape

In [ ]:
l4_df.shape

In [ ]:
l5_df.shape

In [ ]:
l2_df.to_csv(path_or_buf='l2.txt', sep = '\t', index = False)
l3_df.to_csv(path_or_buf='l3.txt', sep = '\t', index = False)
l4_df.to_csv(path_or_buf='l4.txt', sep = '\t', index = False)
l5_df.to_csv(path_or_buf='l5.txt', sep = '\t', index = False)

In [ ]:
# testo....
edge_list = []
for i_l2, l2 in enumerate(l2_df['l2'].tolist()):    
    for w3_be in word_byte_list:
        if l2 & w3 _be == 0:   
            # no letters in common
            existing_letters = l2 | w3_be        
                    
            edge_list.append([l2, w3_be, existing_letters])
    if i_l2 % 10000 == 0:
        print(i_l2)

In [ ]:
def get_word_id(df=pd.DataFrame, cn=str):
    df[(cn + '_id')] = df[cn].map(lc_wid_dict)
    return df

In [ ]:
l2_df = get_word_id(df = l2_df, cn = 'w1')
l2_df = get_word_id(df = l2_df, cn = 'w2')

In [ ]:
l2_df.head()

In [ ]:
l2_df.to_csv(path_or_buf='level2.txt', sep = '\t', index = False)

# BUILD THE L2 CHAR MATRIX

In [ ]:
def populate_level_n_char_matrix(df:pd.DataFrame, cn:str):
    
    # start subtracting A - B. Outcome
    # 1: A has a value, B does not
    # 0: Both A and B do not have the value OR Both A and B have the value
    # -1: A does not have the value, B does

    # build a char matrix
    def pop_cm(letters):
        output = np.zeros(shape = (26, ), dtype = np.int8)
        for l in letters:
            output[letter_dict[l]] += 2
        return output


    cm = np.array(df[cn].map(pop_cm).tolist())

    t1 = ((cm - no_vowel_cm[0, :]) == -1).any(axis = 1).astype(np.int8)
    t2 = ((cm - no_vowel_cm[1, :]) == -1).any(axis = 1).astype(np.int8)
    t3 = ((cm - no_vowel_cm[2, :]) == -1).any(axis = 1).astype(np.int8)

    
    df[cn + '_ve'] = ((t1 + t2 + t3) == 3).astype(int)

    return df


In [ ]:
l2_df = populate_level_n_char_matrix(df = l2_df, cn = 'l2')

In [ ]:
l2_df.head()

In [ ]:
def compute_remaining_vowels(df:pd.DataFrame, curr_level:int):

    # compute remaining vowels
    l_level = 'l' + str(curr_level)
    r_level = 'r' + str(curr_level)


    r_cn =  r_level + '_vowels'    
    l_ve_cn = l_level + '_ve'
    df[(r_cn)] = df[l_level].map(lambda x: len(vowel_set.difference(x)))
    

    # when vowel_escape = 0, we can be more free with the selection mechanism
    # when vowel_escape = 1, we have to be less free. Including the letter y, we have six vowels.
    # That means that if the taboo words are excluded, only 1 word can have two vowels. 
    # to build level 3: when vowel_escape = 1, we need at least 2 vowels
    df = df.loc[(df[r_cn] >= curr_level + 1) | (df[l_ve_cn] == 0) , :].copy()

    # build the extracts
    df[(l_level + '_idx')] = df[l_level].map(lambda x: [letter_dict[l] for l in x])
    df[(r_level + '_idx')] = df[r_level].map(lambda x: [letter_dict[l] for l in x])
    
    return df

In [ ]:
# compute remaining vowels
l2_df = compute_remaining_vowels(df = l2_df, curr_level = 2)

In [ ]:
l2_df.head()

In [ ]:
l2_df.shape

# BUILD LEVEL 3 USING ENUMERATION

In [ ]:
test_l2_df = l2_df.iloc[:10000]

In [ ]:
test_l2_df.head()

In [ ]:
vowel_idx = [letter_dict[l] for l in sorted(vowel_set)]

In [ ]:
test_l2_df.head()

In [ ]:
l3_df_list = []
candidate_words_list = np.full(shape = (100000000, 5), dtype = np.int16,fill_value=-1)
start_pos = 0
for i_row, row in l2_df.iterrows():
    # this is really three rounds of enumeration - all slow
    
    w1, w2, l2, r2, w1_id, w2_id, l2_ve, r2_vowels, l_idx_list, r_idx_list = row

    # these are the words with the existing letters
    #existing_id_list = []
    
    # test_char_matrix = char_matrix[:, l_idx_list] == 1        
    # existing_id_list = word_id_list[test_char_matrix.any(axis = 1)]    
    # remainder_id_list = np.setdiff1d(word_id_list, existing_id_list)    

    # these are words that have at least one remaining letter    
    mr_1_char_matrix = char_matrix[:, r_idx_list] == 1    
    mr_1_id_list = word_id_list[mr_1_char_matrix.any(axis = 1)]
    
    # this is a very expensive operation!
    #remainder_id_list = np.setdiff1d(rem_id_list, ex_id_list)    

    # these are the word ids of candidates 
    if mr_1_id_list.size > 0:                      
    
        # this the char_matrix of remainder words 
        mr_2_char_matrix = char_matrix[mr_1_id_list, :]
        # now, filter out records that have the existing letters
        mr_2_id_list = mr_1_id_list[(mr_2_char_matrix[:, l_idx_list] == 0).all(axis = 1)]
        
        if mr_2_id_list.size > 0:                        

            s1 = mr_2_id_list.shape[0]
            mr_3_char_matrix = char_matrix[mr_2_id_list, :]
            # now, count vowels
            mr_3_id_list = mr_2_id_list[(mr_3_char_matrix[:, vowel_idx].sum(axis = 1) < (r2_vowels - 1 ))]
            s2 = mr_3_id_list.shape[0]
            #print(s1, s2, r2_vowels)

            if mr_3_id_list.size > 0:

                n_candidate_words = mr_3_id_list.shape[0]                  

                output = np.zeros(shape = (n_candidate_words, 5), dtype = np.int16)
                output[:, 0] = w1_id
                output[:, 1] = w2_id
                output[:, 2] = mr_3_id_list         
                output[:, 3] = l2_ve
                output[:, 4] = r2_vowels            

                candidate_words_list[start_pos:start_pos + n_candidate_words, :] = output
            
                start_pos += n_candidate_words

In [ ]:
candidate_words_list.shape

In [ ]:
# remove excess 
candidate_words_list=candidate_words_list[(candidate_words_list != -1).all(axis = 1), :]

In [ ]:
candidate_words_list.shape

In [ ]:
l3_df = pd.DataFrame(candidate_words_list, columns = ['w1_id', 'w2_id', 'w3_id', 'l2_ve', 'r2_vowels'])
l3_df['w1'] = l3_df['w1_id'].map(wid_lc_dict)
l3_df['w2'] = l3_df['w2_id'].map(wid_lc_dict)
l3_df['w3'] = l3_df['w3_id'].map(wid_lc_dict)

In [ ]:
l3_df.head()

In [ ]:
l3_df.shape

In [ ]:
l3_df['l3'] = l3_df[['w1', 'w2', 'w3']].apply(func = lambda x: ''.join(sorted(''.join(x))), axis =1 )

In [ ]:
l3_df.head()

In [ ]:
l3_df['l3'].unique().shape[0]

In [ ]:
l3_df['r3'] = (l3_df['l3'].map(lambda x: ''.join(sorted(lc_set.difference(x)))))
            

In [ ]:
l3_df.shape

In [ ]:
l3_df = l3_df.drop_duplicates(subset=['l3', 'r3'])

In [ ]:
l3_df.shape

In [ ]:
l3_df.head()

In [ ]:
# populate l3 char_matrix

In [ ]:
l3_df['l3_vowel']

In [ ]:
l3_df_list = []
candidate_words_list = []
for i_row, row in test_l2_df.iterrows():
    # this is really three rounds of enumeration - all slow
    
    w1, w2, l2, r2, w1_id, w2_id, l_idx_list, r_idx_list = row

    # these are the words with the existing letters
    #existing_id_list = []
    l_idx_list = [letter_dict[l] for l in l2]        
    test_char_matrix = char_matrix[:, l_idx_list] == 1        
    existing_id_list = word_id_list[test_char_matrix.any(axis = 1)]    
    
    remainder_id_list = np.setdiff1d(word_id_list, existing_id_list)    

    # these are the word ids of candidates 
    if remainder_id_list.size > 0:         
        n_candidate_words = remainder_id_list.shape[0]               
    
        # how can we winnow down the list of candidates?
        r3_word_list = working_word_df.loc[working_word_df['word_id'].isin(remainder_id_list), 'lcase'].tolist()          
        temp_list = [w1, w2, l2, r2, n_candidate_words]
        candidate_words_list.append(temp_list)
        for r3_word in r3_word_list:            
            ex_words = [w1, w2]
            output = get_vowels(letters=l2, r_word = r3_word, min_vowel_count=2, existing_words=ex_words)            
            
            if output:                
                #print(len(output))
                l3_df_list.append(output)




In [ ]:
l3_df = pd.DataFrame(data = l3_df_list, columns = ['w1', 'w2', 'w3', 'l3', 'r3'])

In [ ]:
l3_df.shape

In [ ]:
l3_df.head()

In [ ]:
l3_df.shape

In [ ]:
l3_df = l3_df.drop_duplicates(subset = ['l3', 'r3'])

In [ ]:
l3_df.shape

In [ ]:
cw_df = pd.DataFrame(data = candidate_words_list, columns = ['w1', 'w2', 'l2', 'rr2', 'n_words'])

In [ ]:
cw_df.head()

In [ ]:
test_df.to_csv(path_or_buf='level3.txt', sep = '\t', index = False)

In [ ]:
test_df.head()

# BUILD LEVEL 4 USING ENUMERATION

In [ ]:
l4_df_list = []
for i_row, row in test_df.iterrows():

    w1, w2, w3, l3, r3 = row

    # existing letters
    existing_id_list = []
    for l in l3:
        l_idx = letter_dict[l]   
        test_char_matrix = char_matrix[:, l_idx] == 1
        # print(word_id_list[test_char_matrix])
        existing_id_list.append(word_id_list[test_char_matrix])

    #print(inclusive_id_list)
    existing_id_list = np.concatenate(existing_id_list)        
    
    remainder_id_list = np.setdiff1d(word_id_list, existing_id_list)
    if remainder_id_list.size > 0:                
        #print(l1, r1, outcome_ids.shape)        
        #level2_list.append([l1, r1, existing_id_list.shape[0], remainder_id_list.shape[0]])                       
        #         
        # l1
        # do this later - winnow down the set list sooner
        # do this later - winnow down the set list sooner
        r4_word_list = working_word_df.loc[working_word_df['word_id'].isin(remainder_id_list), 'lcase'].tolist()
        # print(r3_word_list)
        for r4_word in r4_word_list:
            output = get_vowels(letters=l3, r_word = r4_word, min_vowel_count=1, existing_words=[w1, w2, w3])
            if output:
                l4_df_list.append(output)

In [ ]:
l4_df = pd.DataFrame(data = l4_df_list, columns =  ['w1', 'w2', 'w3', 'w4', 'l4', 'r4'])

In [ ]:
l4_df.shape

In [ ]:
l4_df.head()

In [ ]:
test_df = l4_df.drop_duplicates(subset = ['l4', 'r4'])

In [ ]:
test_df.shape

In [ ]:
test_df.to_csv(path_or_buf='level4.txt', index = False, sep =  '\t')

# BUILD LEVEL 5 USING ENUMERATIONS

In [ ]:
l5_df_list = []
for i_row, row in test_df.iterrows():
    #print(i_row)

    w1, w2, w3, w4, l4, r4 = row

    # existing letters
    existing_id_list = []
    for l in l4:
        l_idx = letter_dict[l]   
        test_char_matrix = char_matrix[:, l_idx] == 1
        # print(word_id_list[test_char_matrix])
        existing_id_list.append(word_id_list[test_char_matrix])

    #print(inclusive_id_list)
    existing_id_list = np.concatenate(existing_id_list)        
    
    remainder_id_list = np.setdiff1d(word_id_list, existing_id_list)
    if remainder_id_list.size > 0:                
        #print(l1, r1, outcome_ids.shape)        
        #level2_list.append([l1, r1, existing_id_list.shape[0], remainder_id_list.shape[0]])                       
        #         
        # l1
        # do this later - winnow down the set list sooner
        # do this later - winnow down the set list sooner
        r5_word_list = working_word_df.loc[working_word_df['word_id'].isin(remainder_id_list), 'lcase'].tolist()
        # print(r3_word_list)
        for r5_word in r5_word_list:
            output = get_vowels(letters=l4, r_word = r5_word, min_vowel_count=0, existing_words=[w1, w2, w3, w4])
            if output:
                l5_df_list.append(output)

In [ ]:
l5_df = pd.DataFrame(data = l5_df_list, columns =  ['w1', 'w2', 'w3', 'w4', 'w5', 'l5', 'r5'])

In [ ]:
l5_df.head()

In [ ]:
l5_df.shape

In [ ]:
l5_df = l5_df.drop_duplicates(subset = ['l5', 'r5']).reset_index(drop = True)

In [ ]:
l5_df.head()

In [ ]:
l5_df

In [ ]:
l5_df.to_csv(path_or_buf='level5.txt', sep = '\t', index = False)